# 01 - MCP

In this notebook we'll be exploring the Model Context Protocol (MCP). 
* First we'll cover some background on MCP - what is MCP and why it might be useful to us.
* Then we'll build our very own MCP server and connect it to our agent
* Finally, we'll learn how to connect our agent to the wide library of publicly available MCP servers


## What is Model Context Protocol (MCP)
Anthropic (the makers of MCP) defined MCP as "An open protocol that standardises how your LLM applications connect to and work with your tools and data sources"

Think of MCP as a USB connector that we are very familiar with. It allows you to iterface, say your laptop, with several other perepherals without really worrying about how the connect works - it just works! Without the USB standard, each perepheral would come with it's own adapter, cable etc. A nightmare! In fact, Apple was rather infamously doing just this until the EU forced it to standarize their iPhone chargers (which now default to USB 3 protocols!), which means you can use ant standard charger with your latest iPhone.

Consider another example of how the JDBC/ODBC protocols decoupled applications from databases. Before JDBC/ODBC, if you wanted to switch your application from an Oracle database to an MS SQL database, you often had to rewrite the entire data-access layer of your code because the libraries, function calls, and error handling were completely different. With JDBC/ODBC, you may have to just change the way you connect to the database - rest of the interface API remains the same. Similarly with agents before the advent of MCP - if you wanted to give an AI agent access to "Google Drive," you had to write custom integration code specifically for that agent. If you then moved the "drive" to Microsoft 365, you had to write it all over again. MCP turns "tools" into plug-and-play servers.

Also, consider a custom tool (function) you built for your Agent. Let's say it handles a payment gateway interface enabling your Agent to send & receive payments. Say another developer in your organization (or another organization) wants to enable similar functionality, the she would have to write a similar tool. What if you wrote a MCP server as a standard Payment Gateway? Now suddenly all developers in your organization use the same server to enable their agents with Payments functionality - no code duplication. If your MCP server then goes "public" all developers everywhere can use the same functionality without writing custom tools!

In all the examples above, your application is a _host_ that uses a _client_ (interfacing code - JDBC/ODBC) to interface with the _server_ (Database). 


Now let's drill down to some specifics:
* An MCP host _hosts_ an MCP client, which commun

As a first step, let's build our own MCP server.

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:
from mcp.server.fastmcp import FastMCP
from tavily import TavilyClient
from typing import Dict, Any
from requests import get

In [3]:
mcp = FastMCP("mcp_server")
tavily_client = TavilyClient()

In [5]:
# tool for searching web using MCP protocol
# NOTE the @mcp.tool() decorator rather than our usual @tool decorator
@mcp.tool()
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information based on query"""
    response = tavily_client.search(query)
    return response

In [6]:
# let's define a resource our agent can use
@mcp.resource("github://langchain-ai/langchain-mcp-adapters/blob/main/README.md")
def github_file():
    """resource for accessing the langchain-ai/langchain-mcp-adapters/README.md file"""
    url = "https://raw.githubusercontent.com/langchain-ai/langchain-mcp-adapters/main/README.md"
    try:
        response = get(url)
        return response.text
    except Exception as e:
        print(f"Error occurred while fetching GitHub file: {e}")
        return f"Error occurred while fetching the file -> {str(e)}"

In [ ]:
# define a prompt template
@mcp.prompt()
def prompt():
    """analyze data from langchain-ai repo file with comprehensive insights"""
    return """
    You are a helpful assistant that answers questions about LangChain, LangGraph and LangSmith.
    
    You can use the following tools/resources to answe user's questions:
    - search_web: search the web for information.
    - github_file: Access the langchain-ai repo files.

    If user user asks a question that is NOT RELATED to LangChain, LangGraph or LangSmith, you
    should respond with "I am sorry, I can only answer questions related to LangChain, LangGraph and LangSmith. Please ask a relevant question."

    You may try multiple tools and resource calls to answer the user's question.

    You may also ask clarifying questions to the user to better understand their question before responding.
    """